# 🔍 Production Lakehouse Query

This notebook allows interactive SQL querying of the entire Data Lakehouse (Bronze -> Silver -> Gold).

### Capabilities
1. **Cross-Layer Joins**: Join raw bronze data with aggregated gold metrics.
2. **Time Travel**: Query data as of past timestamps.
3. **Ad-hoc Analysis**: Validate data quality and business logic.

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum as _sum

# Production Credentials
PACKAGES = (
    "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,"
    "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.3_2.12:0.67.0,"
    "software.amazon.awssdk:bundle:2.17.178,"
    "software.amazon.awssdk:url-connection-client:2.17.178,"
    "org.apache.hadoop:hadoop-aws:3.3.1"
)

conf = (pyspark.SparkConf()
    .setAppName('Lakehouse-Query-Pro')
    .set('spark.jars.packages', PACKAGES)
    .set('spark.sql.extensions', 
         'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,'
         'org.projectnessie.spark.extensions.NessieSparkSessionExtensions')
    .set('spark.sql.catalog.nessie', 'org.apache.iceberg.spark.SparkCatalog')
    .set('spark.sql.catalog.nessie.uri', 'http://140.238.224.207:19120/api/v1')
    .set('spark.sql.catalog.nessie.ref', 'main')
    .set('spark.sql.catalog.nessie.authentication.type', 'NONE')
    .set('spark.sql.catalog.nessie.catalog-impl', 'org.apache.iceberg.nessie.NessieCatalog')
    .set('spark.sql.catalog.nessie.warehouse', 's3a://lakehouse-prod/warehouse')
    .set('spark.sql.catalog.nessie.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
    .set('spark.sql.catalog.nessie.s3.endpoint', 'https://bmcfe6z38foz.compat.objectstorage.ap-mumbai-1.oraclecloud.com')
    .set('spark.hadoop.fs.s3a.access.key', '962c9f862226831e4edea90cfcfafb8a8dffcd51')
    .set('spark.hadoop.fs.s3a.secret.key', 'sd2rGU918DTmn35E4xJ8EV7BX2XUt7DkqC8v6WDNDUw=')
    .set('spark.hadoop.fs.s3a.endpoint', 'https://bmcfe6z38foz.compat.objectstorage.ap-mumbai-1.oraclecloud.com')
    .set('spark.hadoop.fs.s3a.path.style.access', 'true')
    .set('spark.hadoop.fs.s3a.connection.ssl.enabled', 'true')
    .set('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem'))

spark = SparkSession.builder.config(conf=conf).getOrCreate()
print("✅ Spark Ready for Production Queries!")

## 📊 List All Available Tables

In [ ]:
spark.sql("SHOW TABLES IN nessie.ecommerce").show(truncate=False)

In [ ]:
# Example: Compare Bronze vs Silver Count
bronze_count = spark.table("nessie.ecommerce.`orders_bronze@bronze`").count()
silver_count = spark.table("nessie.ecommerce.orders_silver").count()

print(f"Raw: {bronze_count:,}")
print(f"Clean: {silver_count:,}")
print(f"Dropped Duplicates: {bronze_count - silver_count:,}")